# 04 — Ce que la grille a dit

**Notebook de cours — `structured-latent-dynamics`**

---

Les notebooks 00 à 03 posent l'instrument : pourquoi l'erreur d'un predictor latent compose
géométriquement (00), la famille `ssm` et sa reparamétrisation bornée (01), les backbones, le
wrapper $\psi/\bar A/\pi$ et le budget équitable (02), la loss JEPA et ses gardes numériques
(03). Aucun n'a encore dit ce que le dépôt a *mesuré* quand ce wrapper a réellement tourné.

Ce notebook le dit. Il referme deux promesses posées au notebook 00 : sa cellule de pont
`§3.5` annonçait que le contrôle D3 et les quatre éliminations concurrentes seraient traités
ici ; sa cellule finale annonçait que l'inventaire des limites du dépôt y deviendrait un
registre tenu à jour. C'est le chapitre le plus dense en chiffres de la série — chaque nombre
qu'il écrit est recoupé à sa source dans `PROTOCOL.md` avant d'être posé sur la page.

## Plan

| § | Contenu |
|---|---|
| 1 | Le problème : des runs propres ne suffisent pas |
| 2 | L'identifiabilité d'abord — le contrôle D3, quatre hypothèses éliminées |
| 3 | Le chemin des campagnes — le pendule, le pivot, le cycle oscillateurs |
| 4 | Le factoriel primaire — six bras, la règle pré-enregistrée, le contraste apparié |
| 5 | La leçon de la base — pourquoi `on-stable-nolift` est pire que les deux autres |
| 6 | Ce qu'on ne sait pas |
| 7 | Ce qui a été prouvé ici, et le pont |

Convention de citation et glossaire (`M4`, `OURegime`, phases, campagnes A/B/C) : ceux du
`notebook 00`, §0.3 et §0.5 — non redéfinis ici.

## 1. Le problème : des runs propres ne suffisent pas

Un run qui termine sans diverger, qui produit un `gate.json` complet, qui ne plante nulle
part — ce n'est pas encore un résultat. Il manque une pièce, et c'est la première que ce
chapitre pose : **une règle de lecture, écrite avant de regarder les nombres**, qui dit à
l'avance ce qui compterait comme un succès, ce qui compterait comme un échec, et ce qu'aucune
mesure ne pourra changer après coup.

C'est la règle **R4** de ce dépôt (`REGLES.md` §1, §4) : *le pré-enregistrement précède
l'exécution qu'il autorise*. Chaque amendement de `PROTOCOL.md` qui déclare une campagne porte
une clause « decided, not derived » — ce qui est **décidé** avant les runs (le seuil, le
critère, la prédiction) est distingué de ce que les runs **mesurent**. Sans cette distinction,
un résultat négatif se répare silencieusement en changeant le seuil après coup ; avec elle, un
résultat négatif devient un fait publiable, aussi digne d'être écrit qu'un résultat positif
(`REGLES.md` §1, hiérarchie correction > honnêteté > lisibilité > vitesse).

Tout ce chapitre suit cette discipline. Chaque table qui suit vient d'une section de
`PROTOCOL.md` dont la première moitié est une déclaration datée, et la seconde une mesure. On
ne mélange jamais les deux au moment de la lecture.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display

np.set_printoptions(precision=4, suppress=True, linewidth=110)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# --- Trois objets repris de notebook 00 (§2.2, §0.6, §3.1), non redérivés ici : leurs preuves
# vivent là-bas, ce notebook les réutilise pour reconstruire le système conducteur (oscillateurs
# couplés à c = 0.10, le régime où le prior rotation-scaling est exactement correct, notebook 00
# §3, dont la trajectoire est déjà tracée en §3.1) sans dupliquer une dérivation déjà faite.
I2 = np.eye(2)
J2 = np.array([[0.0, -1.0], [1.0, 0.0]])
DELTA = 0.05                      # pas d'integration du depot pour les oscillateurs


def A_bar_closed(alpha, omega, dt=DELTA):
    '''exp(dt * (alpha I + omega J)) en forme close (notebook 00, §2.2).'''
    return np.exp(dt * alpha) * (np.cos(dt * omega) * I2 + np.sin(dt * omega) * J2)


def expm_series(M, n_terms=100):
    '''exp(M) par serie de Taylor tronquee : reference independante pour une matrice
    quelconque, y compris hors bloc 2x2 (notebook 00, §0.6 le definit et le verifie).'''
    S = np.eye(M.shape[0])
    T = np.eye(M.shape[0])
    for k in range(1, n_terms + 1):
        T = T @ M / k
        S = S + T
    return S


K = np.array([[1.0 + 0.5, -0.5], [-0.5, 1.5 + 0.5]])   # k1=1, k_c=0.5, k2=1.5 (notebook 00, §3.1)


def A_state(c):
    '''Le generateur 4x4 des deux masses couplees, amortissement visqueux c (notebook 00, §3.1).'''
    Z, Id = np.zeros((2, 2)), np.eye(2)
    return np.block([[Z, Id], [-K, -c * Id]])


print("systeme conducteur : oscillateurs couples, c = 0.10 -> delta(c) = 0 (notebook 00, §3)")

## 2. L'identifiabilité d'abord

Avant de lire une seule campagne du dépôt, une question logiquement antérieure doit être
tranchée : quand le wrapper structuré *apprend* quelque chose, apprend-il la bonne chose ? Une
architecture peut battre des baselines triviales et rester, en même temps, loin de la solution
exacte qu'elle est censée représenter. C'est exactement ce que le contrôle **D3** du dépôt a
mesuré, sur le système conducteur — oscillateurs couplés sous-amortis, `c = 0.10`, régime où
`§3` a établi $\delta(c) = 0$ : le prior rotation-scaling est **exactement correct**.

### 2.1 Le contrôle D3, et ce qu'il mesure

10 cellules (`mlp_k`, `transformer_k` $\times$ 5 seeds), 8000 updates chacune, rollout fermé de
25 pas.

**[MESURÉ — `PROTOCOL.md` §20.0, 10/10 runs complets, zéro divergence]**

| critère | seuil | résultat | médiane | étendue |
|---|---|---|---|---|
| gain vs les deux baselines triviales | $m_{\text{low}} < 1$ | **10/10 PASS** | $0.156$ | $[0.148,\ 0.170]$ |
| erreur d'état à $H=100$ | $< 10^{-3}$ | **0/10** | $0.189$ | $[0.120,\ 0.492]$ |
| erreur spectrale, sur $J_x$ | $< 10^{-3}$ | **0/10** | $0.112$ | $[0.070,\ 0.158]$ |
| erreur d'action, sur $J_u$ | $< 10^{-3}$ | **0/10** | $0.198$ | $[0.141,\ 0.340]$ |

$m_{\text{low}} = 0.156$ veut dire que les modèles battent la persistance et l'extrapolation
linéaire d'environ **32×**, contre le facteur $\gamma = 5$ que le gate exige : ils
**apprennent**. Ils n'**identifient** pas : à 2 décades au-dessus d'un seuil d'exactitude
pré-enregistré, publié avant le run.

In [ ]:
figA = REPO / "assets" / "figA_d3_exactness.png"
if figA.exists():
    display(Image(str(figA)))
else:
    print("assets/figA_d3_exactness.png absent de ce clone — voir la table ci-dessus, "
          "memes chiffres (PROTOCOL.md §20.0, §20.1).")

**Lecture de la figure.** Quatre panneaux, un par ligne de la table ci-dessus, chacun sur sa
propre échelle — délibérément : `PROTOCOL.md` §19.2 interdit de laisser la borne spectrale
résumer les autres. Le panneau du gain est linéaire ; les trois panneaux d'exactitude (état à
$H=100$, erreur spectrale, erreur d'action) sont en échelle logarithmique, parce que la
grandeur qu'ils portent s'étend sur quinze décades entre le gold planté ($1.9\times10^{-15}$)
et le seuil pré-enregistré ($10^{-3}$). Chaque point est une des dix cellules (`mlp_k` ou
`transformer_k` $\times$ 5 seeds), le trait horizontal leur médiane ; les trois repères
verticaux — gold, moindres carrés, seuil — sont des ancres analytiques de `§20.1`, jamais une
mesure de ces runs. Les points se logent entre $110\times$ et $200\times$ au-dessus du seuil :
l'écart n'est pas la lecture d'un chiffre isolé, c'est un écart de régime sur l'ensemble des
quinze décades.

**Ce qu'elle ne montre pas.** Le $110\times$-$200\times$ affiché est la distance **au seuil**,
pas aux moindres carrés — sur l'erreur spectrale, le rapport aux moindres carrés vaut environ
$0.112 / 2.11\times10^{-8} \approx 5.3\times10^{6}\times$ (les deux médianes déjà citées en
`§2.1` et `§2.2`), un chiffre que cette figure ne trace nulle part et qu'il ne faut pas lire sur
son échelle sans le recalculer.

### 2.2 Quatre explications concurrentes, chacune éliminée par sa propre mesure

**[MESURÉ — `PROTOCOL.md` §20.1]**

1. **Ce ne sont pas les données.** Les moindres carrés ordinaires, sur le *même* split
   d'entraînement, récupèrent l'opérateur : erreur spectrale $2.11\times10^{-8}$, erreur
   d'action $1.58\times10^{-7}$, rollout libre à 100 pas $e_{100} = 7.98\times10^{-14}$. Les
   trois franchissent la borne $10^{-3}$ de cinq à onze décades. *(Recoupement indépendant :
   l'erreur spectrale des moindres carrés, $2.1141\times10^{-8}$, tombe exactement sur l'écart
   RK4-contre-ZOH-exacte que `§19.7` avait déclaré, $2.114\times10^{-8}$ — les moindres carrés
   récupèrent le propagateur RK4 à la précision machine, et son résidu **est** l'écart de
   l'intégrateur, confirmé par un chemin entièrement indépendant de celui qui l'a déclaré.)*
2. **Ce n'est pas l'architecture.** Le gold de la phase C, paramètres plantés à la main,
   reproduit la solution ZOH exacte à $1.9\times10^{-15}$ : l'optimum existe **dans** la classe
   d'hypothèses que les runs ont utilisée.
3. **Ce n'est pas le budget, à aucune échelle proportionnée.** La décroissance de la courbe de
   validation passe de $-1.66$ décade pour 1000 updates (époques 0-1000) à $-0.023$
   (époques 6000-8000). Fermer l'écart de 1.24 décade restant au taux terminal prendrait
   **50 000 à 100 000 updates supplémentaires** — 7 à 13 fois le budget déclaré — sous une
   extrapolation log-linéaire que de telles courbes aplatissent plutôt qu'elles n'honorent.
4. **Ce n'est pas la sur-paramétrisation état-dépendante.** L'hypothèse que la tête
   contextuelle dépense sa liberté à sur-ajuster une vérité LTI a été **testée et réfutée** :
   forcer $W_\theta = 0$ (l'opérateur exactement constant, la forme LTI) rend les **deux**
   bornes *pires* — spectrale $0.130 \to 0.190$, action $0.138 \to 0.947$. La dépendance au
   contexte apprise porte du signal réel.

**L'énoncé, et sa portée.** Sous le régime déclaré — AdamW à $\text{lr}=10^{-3}$, 8000
updates, lot de 8 épisodes, la loss latente de style JEPA avec $h_{fr}=25$ — la descente de
gradient converge vers une solution environ **200×** pire que celle que les moindres carrés
ordinaires trouvent en forme close sur les mêmes données. Le seuil n'est pas mal spécifié :
`§20.1` établit qu'il est atteignable, et il reste exactement où il a été fixé.

### 2.3 Une miniature — et ce qu'elle peut prouver

Ce qui suit est **[ILLUSTRATIF]**. Ce n'est pas une reproduction de D3 : pas le même encodeur
(il n'y en a pas), pas la même tête contextuelle, pas la même taille, pas les mêmes données.
C'est une **maquette du mécanisme**, construite pour répondre à une question précise et
falsifiable :

> *Quel est l'ingrédient minimal qui fait passer un problème exactement résoluble en forme
> close à un problème que la descente de gradient ne résout pas ?*

On procède par élimination, comme en `§2.2`, sur une maquette où l'on peut allumer les
ingrédients un par un. Cas de référence d'abord : les moindres carrés.

In [ ]:
# --- [ILLUSTRATIF] Le systeme, ses donnees, et la solution en forme close ------------------
C_DAMP = 0.10                                    # regime ou delta(c) = 0 : prior EXACT
A_cont = A_state(C_DAMP)
A_true = expm_series(DELTA * A_cont)             # operateur discret ZOH exact, 4 x 4

ev_disc = np.linalg.eigvals(A_true)
print("rayon spectral discret r =", max(abs(ev_disc)).round(8))
print("spectre discret (arrondi) :", np.round(np.sort_complex(ev_disc), 5))
print(f"c = {C_DAMP} : regime delta(c) = 0, le prior rotation-scaling est EXACT (notebook 00, §3)\n")

rng = np.random.default_rng(7)
N_EP, T_EP = 60, 200
X_list, Y_list = [], []
for _ in range(N_EP):
    x = 0.5 * rng.standard_normal(4)
    for _ in range(T_EP):
        xn = A_true @ x
        X_list.append(x); Y_list.append(xn)
        x = xn
X = np.array(X_list); Y = np.array(Y_list)
print(f"donnees : {len(X)} transitions, dimension d'etat 4 (donnees NON bruitees)")

# --- reference : moindres carres ordinaires (forme close) ----------------------------------
A_ls = np.linalg.lstsq(X, Y, rcond=None)[0].T
print(f"\n[reference] moindres carres : |A_ls - A_true|_F = {np.linalg.norm(A_ls - A_true):.3e}")

x0 = 0.5 * np.random.default_rng(123).standard_normal((256, 4))
xs_true, xs_ls = x0.copy(), x0.copy()
for _ in range(100):
    xs_true = xs_true @ A_true.T
    xs_ls = xs_ls @ A_ls.T
print(f"[reference] moindres carres : e_100 (rollout libre) = "
      f"{np.linalg.norm(xs_ls - xs_true, axis=1).mean():.3e}")
print(f"[reference] conditionnement de la matrice de Gram : "
      f"{np.linalg.cond(X.T @ X):.3f}")

**Lecture.** Les moindres carrés récupèrent l'opérateur exact à $10^{-15}$ près et le
rollout libre à 100 pas dérive de l'ordre de $10^{-14}$ : sur ce jeu, la solution n'est pas
seulement *dans* la classe, elle est **calculable en une ligne**. Le conditionnement de la
matrice de Gram vaut environ $2.4$ : le problème est **très bien conditionné**. Retenons ce
chiffre, il va servir à éliminer une explication.

*(L'ordre de grandeur de notre $e_{100}$ est voisin du $7.98\times10^{-14}$ mesuré en `§20.1`.
C'est une coïncidence de deux problèmes bien conditionnés à 4 états, pas une reproduction, et
il ne faut rien en tirer de plus.)*

### 2.4 Hypothèse A — « la descente de gradient est trop faible »

C'est l'explication réflexe. Testons-la sur la **même** paramétrisation structurée que le
dépôt : deux blocs rotation-scaling $\bar A(\alpha,\omega)$, un lift linéaire $P$ et une
projection linéaire $Q$, prédiction $\hat z_{t+1} = Q\,\bar A\,P\,z_t$. Objectif : erreur à
**un pas**. Les gradients sont analytiques (pas de différences finies) et se dérivent de la
forme close de `notebook 00 §2.2` :

$$\frac{\partial \bar A_i}{\partial \alpha_i} = \Delta\,\bar A_i,
\qquad
\frac{\partial \bar A_i}{\partial \omega_i} = \Delta\,\bar A_i J,$$

vérifiées contre des différences finies avant tout entraînement : un gradient faux produit un
joli plateau qu'on interpréterait comme un résultat.

In [ ]:
# --- [ILLUSTRATIF] Cellule structuree lineaire + Adam, gradients analytiques ---------------
R_BLOCKS = 2
DZ = 4


def A_blocks(alpha, omega, dt=DELTA):
    '''Transition bloc-diagonale : r blocs rotation-scaling exp(dt (alpha_i I + omega_i J)).'''
    M = np.zeros((2 * R_BLOCKS, 2 * R_BLOCKS))
    for i in range(R_BLOCKS):
        M[2 * i:2 * i + 2, 2 * i:2 * i + 2] = A_bar_closed(alpha[i], omega[i], dt)
    return M


def block_grads(dA, alpha, omega, dt=DELTA):
    '''d(loss)/d(alpha_i), d(loss)/d(omega_i) a partir de d(loss)/d(Abar).'''
    ga, gw = np.zeros(R_BLOCKS), np.zeros(R_BLOCKS)
    for i in range(R_BLOCKS):
        g = dA[2 * i:2 * i + 2, 2 * i:2 * i + 2]
        B = A_bar_closed(alpha[i], omega[i], dt)
        ga[i] = dt * np.sum(g * B)              # dAbar/dalpha = dt * Abar
        gw[i] = dt * np.sum(g * (B @ J2))       # dAbar/domega = dt * Abar J
    return ga, gw


_a = np.array([-0.21, 0.13]); _w = np.array([1.7, -0.9]); _h = 1e-6
_W = np.random.default_rng(3).standard_normal((2 * R_BLOCKS, 2 * R_BLOCKS))
_f = lambda a, w: float(np.sum(_W * A_blocks(a, w)))
_ga, _gw = block_grads(_W, _a, _w)
_fd_a = np.array([(_f(_a + _h * np.eye(2)[i], _w) - _f(_a - _h * np.eye(2)[i], _w)) / (2 * _h)
                  for i in range(2)])
_fd_w = np.array([(_f(_a, _w + _h * np.eye(2)[i]) - _f(_a, _w - _h * np.eye(2)[i])) / (2 * _h)
                  for i in range(2)])
print("controle de gradient (analytique vs differences finies centrees) :")
print(f"  d/d alpha : ecart relatif max = {np.abs(_ga - _fd_a).max() / np.abs(_fd_a).max():.2e}")
print(f"  d/d omega : ecart relatif max = {np.abs(_gw - _fd_w).max() / np.abs(_fd_w).max():.2e}")


class Adam:
    def __init__(self, params, lr):
        self.p, self.lr = params, lr
        self.m = [np.zeros_like(x) for x in params]
        self.v = [np.zeros_like(x) for x in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for j, (x, g) in enumerate(zip(self.p, grads)):
            self.m[j] = 0.9 * self.m[j] + 0.1 * g
            self.v[j] = 0.999 * self.v[j] + 0.001 * g ** 2
            mh = self.m[j] / (1 - 0.9 ** self.t)
            vh = self.v[j] / (1 - 0.999 ** self.t)
            x -= self.lr * mh / (np.sqrt(vh) + 1e-8)


def init_params(seed):
    r = np.random.default_rng(seed)
    return [0.1 * r.standard_normal(R_BLOCKS),
            r.standard_normal(R_BLOCKS),
            r.standard_normal((2 * R_BLOCKS, DZ)) / 2,
            r.standard_normal((DZ, 2 * R_BLOCKS)) / 2]


def train_one_step(seed, iters=2000, lr=1e-2, batch=None):
    '''Objectif a UN pas : || Q Abar P z_t - z_{t+1} ||^2.'''
    alpha, omega, P, Q = params = init_params(seed)
    opt = Adam(params, lr)
    r = np.random.default_rng(10_000 + seed)
    n = len(X)
    for _ in range(iters):
        if batch is None:
            Xb, Yb = X, Y
        else:
            idx = r.integers(0, n, batch); Xb, Yb = X[idx], Y[idx]
        Ab = A_blocks(alpha, omega)
        M = Q @ Ab @ P
        E = Xb @ M.T - Yb
        dM = 2.0 / len(Xb) * (E.T @ Xb)
        dQ = dM @ (Ab @ P).T
        dA = Q.T @ dM @ P.T
        dP = (Q @ Ab).T @ dM
        ga, gw = block_grads(dA, alpha, omega)
        opt.step([ga, gw, dP, dQ])
    return Q @ A_blocks(alpha, omega) @ P


print("\n[ILLUSTRATIF] Hypothese A : Adam sur la loss a UN pas, parametrisation structuree")
print(f"{'variante':>26} | {'|M - A_true|_F, par seed':>40}")
print("-" * 72)
for label, kw in [("plein lot", dict(batch=None)), ("mini-lot 256", dict(batch=256)),
                  ("mini-lot 32", dict(batch=32))]:
    errs = [np.linalg.norm(train_one_step(s, **kw) - A_true) for s in range(3)]
    print(f"{label:>26} | {'  '.join(f'{e:.2e}' for e in errs):>40}")

**Lecture — l'hypothèse A est réfutée par notre propre maquette.** Sur l'objectif à un
pas, Adam atteint la **précision machine**, quel que soit le lot. La descente de gradient n'a
aucune difficulté intrinsèque avec cette paramétrisation, ni avec sa non-convexité (le produit
$Q\bar A P$ l'est pourtant), ni avec le bruit de mini-lot — parce que les données sont **sans
résidu** (régime d'interpolation), et `§20.1` mesure que celles du dépôt le sont aussi presque
autant (moindres carrés à $2.11\times10^{-8}$).

**Ce qu'on ne peut pas conclure.** Que le run réel n'ait pas eu de problème d'optimiseur : nous
n'avons ni son échelle, ni ses MLP $\psi/\pi$ non linéaires, ni sa tête contextuelle. La
maquette établit que **ces trois ingrédients-là ne suffisent pas** à créer l'écart. Il faut
chercher ailleurs.

### 2.5 Hypothèse B — c'est le régime d'entraînement, pas l'optimiseur

Le dépôt n'entraîne pas sur une erreur à un pas. Depuis l'amendement 6, `h_fr = 25` : la perte
se calcule sur un **rollout fermé de 25 pas depuis l'initialisation** — le modèle consomme ses
propres prédictions, sans réinjection d'observation, dès le premier pas d'entraînement.
`notebook 00 §1.3` a montré ce que cela change : le gradient traverse $H$ applications de $\bar
A$, multiplié par $\sim r^H$ avec $r = e^{\Delta\alpha}$. À l'initialisation, $\alpha$ est
aléatoire et **non borné** (`PROTOCOL.md` §13.2 laisse `alpha` libre). Changeons
**uniquement** cet ingrédient — même modèle, mêmes gradients analytiques, même optimiseur,
même budget.

In [ ]:
# --- [ILLUSTRATIF] Meme modele, meme optimiseur : objectif rollout FERME a H pas ------------
H_ROLL = 25
N_TRAJ_TR = 400

rng_tr = np.random.default_rng(11)
Z0 = 0.5 * rng_tr.standard_normal((N_TRAJ_TR, DZ))
TRAJ = np.zeros((N_TRAJ_TR, H_ROLL + 1, DZ))
TRAJ[:, 0] = Z0
for t in range(H_ROLL):
    TRAJ[:, t + 1] = TRAJ[:, t] @ A_true.T


def train_rollout(seed, iters=2000, lr=1e-2, batch=128, H=H_ROLL):
    '''Objectif rollout FERME : h_0 = P z_0, h_{t+1} = Abar h_t, perte sur t = 1..H.'''
    alpha, omega, P, Q = params = init_params(seed)
    opt = Adam(params, lr)
    r = np.random.default_rng(20_000 + seed)
    g0 = None
    for _ in range(iters):
        idx = r.integers(0, N_TRAJ_TR, batch)
        tj = TRAJ[idx]
        Ab = A_blocks(alpha, omega)
        Hs = [tj[:, 0] @ P.T]
        for _t in range(H):
            Hs.append(Hs[-1] @ Ab.T)
        dQ = np.zeros_like(Q); dAb = np.zeros((2 * R_BLOCKS, 2 * R_BLOCKS))
        gH = [np.zeros((batch, 2 * R_BLOCKS)) for _ in range(H + 1)]
        scale = 2.0 / (batch * H)
        for t in range(1, H + 1):
            E = scale * (Hs[t] @ Q.T - tj[:, t])
            dQ += E.T @ Hs[t]
            gH[t] += E @ Q
        for t in range(H, 0, -1):                        # BPTT exacte (modele lineaire)
            dAb += gH[t].T @ Hs[t - 1]
            gH[t - 1] += gH[t] @ Ab
        dP = gH[0].T @ tj[:, 0]
        ga, gw = block_grads(dAb, alpha, omega)
        grads = [ga, gw, dP, dQ]
        if g0 is None:
            g0 = np.sqrt(sum(float(np.sum(g ** 2)) for g in grads))
        opt.step(grads)
        if not np.all(np.isfinite(np.concatenate([np.ravel(x) for x in params]))):
            return np.nan, g0, np.nan
    Ab = A_blocks(alpha, omega)
    return np.linalg.norm(Q @ Ab @ P - A_true), g0, float(np.exp(DELTA * alpha).max())


# On balaie aussi le taux d'apprentissage : PROTOCOL.md §21.2 nomme l'asymetrie de taux comme
# un confondant, il serait malhonnete de conclure sur un seul taux.
print("[ILLUSTRATIF] Hypothese B : seul l'objectif change. Mediane sur 3 seeds de |M - A_true|_F")
print(f"{'lr':>8} | {'objectif UN pas':>17} | {'objectif ROLLOUT 25':>20} | {'rapport':>10}")
print("-" * 66)
summary = {}
for lr in [1e-3, 3e-3, 1e-2]:
    e1 = np.median([np.linalg.norm(train_one_step(s, iters=2000, lr=lr, batch=256) - A_true)
                    for s in range(3)])
    eH = np.median([train_rollout(s, iters=2000, lr=lr)[0] for s in range(3)])
    summary[lr] = (e1, eH)
    print(f"{lr:8.0e} | {e1:17.2e} | {eH:20.2e} | {eH / e1:10.1e}")

best_1 = min(v[0] for v in summary.values())
best_H = min(v[1] for v in summary.values())
print("\nrecapitulatif [ILLUSTRATIF] — chaque objectif a son meilleur taux parmi les 3 testes :")
print(f"  moindres carres (forme close)          : {np.linalg.norm(A_ls - A_true):.2e}")
print(f"  Adam, objectif a UN pas                : {best_1:.2e}")
print(f"  Adam, objectif ROLLOUT ferme 25 pas    : {best_H:.2e}")
print(f"  ecart entre les deux objectifs         : {best_H / best_1:.1e} x")

**Lecture.** Un seul ingrédient a changé — l'objectif passe de « un pas » à « rollout
fermé de 25 pas » — et l'écart au minimiseur exact passe de la précision machine à
$\sim 10^{-3}$, soit **treize ordres de grandeur**. Modèle, gradients, optimiseur, budget,
données : identiques.

**Ce qu'on ne peut pas conclure, et il faut être net.** Cette maquette ne reproduit pas D3 —
pas d'encodeur, pas de MLP non linéaires, pas de tête contextuelle, pas la même échelle. Elle
ne démontre pas que le rollout fermé soit *la* cause dans le run réel ; elle démontre qu'il
**suffit**, sur une maquette où tout le reste est contrôlé, à produire un écart de la bonne
nature. Elle isole un candidat testable, pas une preuve.

**Ce que la maquette établit.** Ni la non-convexité, ni le bruit de mini-lot, ni le mauvais
conditionnement ne suffisent à empêcher la descente de gradient d'atteindre la précision
machine sur ce problème. Le **régime d'entraînement**, lui, y suffit à lui seul. C'est ce
candidat que le chemin des campagnes, section suivante, va retrouver — mesuré cette fois, pas
simulé.

## 3. Le chemin des campagnes

Le contrôle D3 a établi l'identifiabilité sur les seuls bras structurés, sans bras `off` en
face : `PROTOCOL.md` §19.1 le dit explicitement, *« D3 compares no arms »*. Le contraste qui
donne son sujet à ce dépôt — la structure paie-t-elle ? — restait donc à mesurer. Le chemin
pour y arriver n'est pas allé en ligne droite. C'est une suite de questions, chacune posée
parce que la précédente a buté sur quelque chose.

### 3.1 D4 : le pendule, et une conclusion qui ne se lit pas

**[MESURÉ — `PROTOCOL.md` §21.0, 40/40 runs complets, zéro divergence]**. `DampedPendulum` —
un système **non linéaire**, où un prior LTI est faux par construction — croisé avec
`OURegime`, quatre arms (`mlp`, `transformer`, `mlp_k`, `transformer_k`), cinq seeds :

| stratum | arm | `e_1` | `m_low` | `m_high` | **apprend** |
|---|---|---|---|---|---|
| OU | `mlp` | 0.0106 | 0.505 | 0.540 | **5/5** |
| OU | `transformer` | 0.0229 | 0.723 | 1.172 | 1/5 |
| OU | `mlp_k` | 0.0520 | 1.148 | 2.656 | 0/5 |
| OU | `transformer_k` | 0.0492 | 1.112 | 2.516 | 0/5 |
| AC | `mlp` | 0.0119 | 0.966 | 0.777 | 3/5 |
| AC | `transformer` | 0.0297 | 0.854 | 1.934 | 0/5 |
| AC | `mlp_k` | 0.0438 | 1.690 | 2.857 | 0/5 |
| AC | `transformer_k` | 0.0436 | 1.502 | 2.841 | 0/5 |

**Une cellule sur huit passe.** Après une mesure qui a montré que la borne haute n'était pas
neutre à l'arm (`§21.1`, une conclusion elle-même retirée ensuite en `§22.0` faute d'être
soutenue par sa propre table) et la correction de l'asymétrie de taux qu'elle a exposée
(`§21.2`-`§21.3`), le verdict pré-enregistré reste :

> **[MESURÉ — `PROTOCOL.md` §21.4]** *« M4 is not crossed. »* Sept cellules sur huit ne passent
> pas le gate de contrôle positif. Aucune location GPU et aucune revendication comparative de
> phase E n'est autorisée par cette section.

`M4` est glosé au glossaire commun (`notebook 00`, §0.5) : il porte sur `DampedPendulum`, le
système que ce chapitre quitte dès la section suivante, pas sur le résultat central de cette
série — celui-ci vit sur `CoupledOscillators`, à partir de `§3.2`.

In [ ]:
figB = REPO / "assets" / "figB_horizon.png"
if figB.exists():
    display(Image(str(figB)))
else:
    print("assets/figB_horizon.png absent de ce clone — voir la table ci-dessus, "
          "memes chiffres (PROTOCOL.md §21.0, §21.1, §21.2).")

**Lecture de la figure.** La table ci-dessus ne donne que $e_1$, $m_{\text{low}}$ et
$m_{\text{high}}$ — trois nombres par cellule, à un seul horizon de lecture. La figure relit les
mêmes 40 artefacts au complet : les deux premiers panneaux tracent $e_h$ pour $h=1$ à $25$, un
panneau par strate (`OU`, `AC`), trait plein pour `structure: on`, pointillé pour `off`, les cinq
graines en fines lignes derrière la médiane épaisse. Le troisième panneau lit le taux de
composition $\rho = e_{25}/e_1$ par cellule, cinq points par arm. Dans 3 des 4 paires
(backbone, strate), le bras structuré part d'une erreur à un pas plus grande et la compose
plus lentement que son homologue `off` — visible dans la forme des courbes, pas seulement dans
un chiffre au gate.

**Ce qu'elle ne montre pas.** Deux confondants, ni l'un ni l'autre réparable depuis ces
artefacts : l'asymétrie de taux d'apprentissage déjà nommée ci-dessus (`§21.2`-`§21.3`), et le
fait que `structure: on` change trois choses à la fois — la transition, le lift $\psi$, la
projection $\pi$ — jamais une seule. La figure ne dit donc pas que la structure *cause* la
composition plus lente ; elle montre que le signe mesuré au gate a une forme, sur tout
l'horizon, et pas seulement une valeur à $H=25$.

### 3.2 Le pivot, par dérogation écrite

Le seul système où $\delta(c) = 0$ — le prior est exactement correct — est `CoupledOscillators`,
et D3 y avait justement retiré tout bras `off`. Le contraste que ce dépôt existe pour mesurer
n'avait donc, à ce stade, **aucune mesure sur le système où le prior est vrai**. Continuer sur
le pendule aurait mesuré une question différente ; le dépôt pivote, et le fait par écrit avant
d'exécuter quoi que ce soit :

> **[MESURÉ — `PROTOCOL.md` §23.4]** *« These roots have NO gate. »* La règle des cinq passes
> individuelles (`§12.4`) est une règle de **gate** ; elle ne s'applique pas ici, faute de gate
> à passer. Une médiane peut s'AFFICHER dans une figure ; elle ne décide rien. La clause de
> `§21.4` qui interdisait toute revendication comparative de phase E est **levée pour ces trois
> racines, et pour elles seules**, datée, par cette section.

Trois racines, déclarées ensemble avant qu'aucune ne tourne, chacune **auto-suffisante** — elle
rejoue ses propres cellules de référence, aucune lecture croisée entre racines (`§14.6`) :

| racine | campagne | cellules |
|---|---|---|
| `results_ssm_delta/` | C — la famille `ssm` a-t-elle un taux entraînable ? | 20 |
| `results_osc_n/` | A — le gain contre la taille d'échantillon, prior exact | 45 (+15 conditionnelles) |
| `results_osc_delta/` | B — le gain contre la mauvaise spécification du prior | 45 |

Le facteur `structure` y prend **trois** niveaux, pas deux : `off`, `on-free` (le rayon libre
de `§13.2`) et `on-stable` — une reparamétrisation qui borne le rayon dans $(0,1)$, motivée
hors de la littérature des world models (SE-adLIF, Balanced Resonate-and-Fire, LinOSS) où un
rayon non borné diverge sous rollout long ; la forme symplectique-Euler alternative a été
**testée et rejetée** (rayon mesuré jusqu'à $368$ hors du régime oscillatoire). C'est un
troisième niveau mesuré à côté de `on-free`, jamais un correctif qui lui est appliqué.

### 3.3 Le cycle oscillateurs : 164 cellules, et une régularité qui pourrait être un artefact

**[MESURÉ — `PROTOCOL.md` §23-§25]**. Les trois racines ci-dessus tournent, puis deux
questions supplémentaires se déclarent à leur tour — `results_osc_hfr/` (le régime
d'entraînement lui-même, l'ingrédient que la maquette de la `§2.5` a isolé) et
`results_osc_decomp/` (la décomposition transition/lift). *« No amount of further analysis of
the existing 164 cells produces »* la question qui reste ouverte à ce stade — laquelle, la
`§4` de ce chapitre le dit. Deux résultats de ce cycle méritent d'être lus ici avant d'y
arriver.

**Le régime d'entraînement, mesuré et non plus simulé.** `results_osc_hfr/`, 20/20 cellules,
zéro divergence : deux niveaux de `h_fr` (5 et 25), `off` contre `on-stable`, cinq seeds.

| `h_fr` | arm | médiane $m_{\text{low}}$ | IQR |
|---|---|---|---|
| 5 | `off` | **0.1184** | $[0.1174,\ 0.1388]$ |
| 5 | `on-stable` | **3.1302** | $[2.2951,\ 3.1925]$ |
| 25 | `off` | **0.2624** | $[0.2209,\ 0.2760]$ |
| 25 | `on-stable` | **0.9029** | $[0.6149,\ 0.9183]$ |

**[MESURÉ — `PROTOCOL.md` §25.8]** Le signe est **préservé** aux deux horizons —
`on-stable` reste pire que `off` (ratio $26.44\times$ à `h_fr=5`, $3.44\times$ à `h_fr=25`) —
donc le négatif n'est pas un artefact du régime d'entraînement particulier que `§2.5` a isolé
comme candidat suffisant : il **survit** au changement d'horizon. Il devient à ce titre, dans
les mots de `§25.2`, le résultat principal du dépôt. Une observation reste sans explication
poursuivie : les deux bras se déplacent en sens **opposés** quand l'horizon raccourcit
(`off` s'améliore $2.2\times$, `on-stable` se dégrade $3.5\times$) — une interaction nommée,
non chassée, la question s'arrête là par construction du plafond de cette racine (`§25.5`).

**La question que la décomposition a posée, et sa réponse en `§5`.** Une deuxième racine
(`results_osc_decomp/`, 30 cellules) demande si la pénalité structurée vient de la
**transition** apprise ou du couple lift/projection $\psi/\pi$ qui l'accompagne, en ajoutant un
troisième bras, `on-stable-nolift` — la transition structurée agissant directement sur $z$,
avec $\psi = \pi = \mathrm{identité}$. La règle de lecture, écrite avant les nombres
(`§25.3`), attendait ce bras entre `off` et `on-stable`. **[MESURÉ — `PROTOCOL.md` §26.0]** il
atterrit **au-dessus des deux**, $3.1\times$ pire que `on-stable` à $n=8$, $13.1\times$ à
$n=256$ — hors de la partition prévue. Pourquoi, c'est la question de la section 5.

## 4. Le factoriel primaire

Le cycle oscillateurs a couru sur **un** backbone. Vérifié par balayage des fichiers de
configuration des six racines : `mlp` et `mlp_k`, rien d'autre — `transformer` n'a jamais
tourné sur `CoupledOscillators`, `transformer_k` y a tourné cinq fois comme contrôle D3,
jamais comme comparaison. Le résultat principal de ce dépôt — la structure explicite ne paie
pas sur le système où le prior est exact — repose donc sur **un** backbone, quand la
comparaison que ce dépôt existe pour faire croise `mlp`, le minimal causal transformer
(`transformer`) et le Mamba-style minimal selective SSM (`ssm`) avec `off`/`on`
(`notebook 02` §2). C'est le trou que `PROTOCOL.md` §27 ferme, par une dérogation écrite au
plafond de `§25.5` : *« Decided, not derived »*, datée avant le premier run.

### 4.1 `results_primary_factorial/`, 30 cellules

Une stratum, six arms (`mlp`, `mlp_k`, `transformer`, `transformer_k`, `min_ssm`, `ssm_k`),
cinq seeds. Budget vérifié à $\pm5\%$ (`notebook 02` §4 le recalcule et le recoupe déjà — non
reproduit ici). Chaque backbone entre au taux déjà connu pour lui entraîner sur ce système
(`mlp` par l'argmin de `§23.3`, `transformer` par le contrôle D3, `ssm` par `§24.0`) : aucun
taux n'a été balayé sur cette grille elle-même, et le confondant — un ordre entre backbones ne
peut pas séparer « ce backbone est meilleur ici » de « son taux hérité lui convient mieux » —
est nommé avant les runs plutôt que découvert après (`§27.2`).

**[MESURÉ — `PROTOCOL.md` §27.8, 30/30 cellules complètes, zéro divergence]**. Statistique
primaire $e_H$ à $H=25$ (notebook 03), plus bas est mieux :

| famille | $e_H$ moyen | SE | $m_{\text{low}}$ médian | verdict du gate |
|---|---|---|---|---|
| `mlp` | **0.0702** | 0.0040 | 0.2624 | `learns` |
| `transformer` | 0.1197 | 0.0106 | 0.4464 | `learns` |
| `mlp_k` | 0.2153 | 0.0329 | 0.9029 | `does_not_learn` |
| `min_ssm` | 0.3353 | 0.0542 | 1.4218 | `does_not_learn` |
| `transformer_k` | 0.3681 | 0.0673 | 1.3101 | `does_not_learn` |
| `ssm_k` | **0.6189** | 0.2021 | 1.5229 | `does_not_learn` |

*(Une précision datée, elle-même corrigée en cours de route : `§27.3` avait d'abord déclaré le
VPT comme statistique primaire. Avant qu'un seul `gate.json` de cette racine n'existe,
`§27.7` a mesuré que 87.6% des épisodes des cinq racines précédentes sont censurés à
`H=25` — la médiane de $e_H$ y vaut $0.1592$, trois fois sous le seuil $\tau=0.5$ du VPT — et a
corrigé la statistique primaire en $e_H$ avant qu'aucune donnée de cette racine ne soit lue. Le
VPT reste publié, avec son statut de censure ; il ne porte plus les verdicts.)*

---

### 4.2 La lecture pré-enregistrée, et une règle qu'on ne répare pas après l'avoir vue casser

> **[MESURÉ — `PROTOCOL.md` §27.8.1]** **Une relation `contradicted`, huit `indeterminate`, une
> `not_tested`.** `transformer > mlp` est contredite : `transformer` est *pire*
> ($+0.0496 \pm 0.0097$). Les huit autres relations pré-enregistrées impliquent chacune un bras
> non éligible sous la règle de `§27.3` — 20 des 30 runs portent le verdict de gate
> `does_not_learn` — et cette règle envoie toute comparaison qui en dépend vers `indeterminate`.
>
> Ce résultat mérite qu'on s'y arrête, parce que la règle qui le produit s'est révélée être le
> mauvais instrument pour cette grille : elle confond « la mesure est valide » (30 cellules sur
> 30 le sont) et « le contrôle positif à deux bornes est passé » (seulement 10). **La règle n'a
> pas été déplacée pour obtenir un résultat.** Elle est laissée exactement où elle a été écrite,
> et la lecture qu'elle produit — une relation sur neuf — est publiée comme le verdict primaire
> (`§27.8.5`). Une lecture exploratoire, post-hoc et étiquetée comme telle, existe à côté
> (`§27.8.4`) : sous une éligibilité définie sur la validité de la mesure plutôt que sur le
> contrôle positif, les neuf relations s'inversent et sont toutes `contradicted` — mais elle ne
> porte **aucun** poids confirmatoire et ne remplace aucun verdict de `§27.8.1`.
>
> **Pourquoi ne pas réparer la règle.** Une règle de lecture pré-enregistrée qui se révèle
> mal calibrée *après* avoir vu les nombres est, par construction, une règle qu'on ne peut plus
> corriger sans soupçon : rien ne distingue alors une correction honnête d'un seuil déplacé pour
> faire apparaître un résultat. `§20.1`, `§27.7` (le VPT) et cette section elle-même tracent la
> même ligne : une erreur trouvée **avant** que la donnée existe se répare (un amendement) ; une
> erreur trouvée **après** se publie et s'enregistre à côté (une observation), jamais en silence,
> jamais réparée rétroactivement. C'est `REGLES.md` §1 — correction avant honnêteté, honnêteté
> avant lisibilité — appliqué à la règle elle-même, pas seulement à ses résultats.

---

### 4.3 Le contraste apparié, seul lu sans confondant

Chaque paire `on`/`off` d'un même backbone partage son taux, son jeu de données, sa seed et son
budget — c'est la seule comparaison de cette grille libre du confondant de `§27.2` :

| backbone | contraste `on − off` | SE | signes (5 seeds) |
|---|---|---|---|
| `mlp` | **+0.1451** | 0.0351 | 5+/0− |
| `transformer` | **+0.2484** | 0.0670 | 5+/0− |
| `ssm` | +0.2836 | 0.2379 | 3+/2− |

**[MESURÉ — `PROTOCOL.md` §27.9]** Positif sur `mlp`, `transformer` et `ssm` en moyenne — mais
pas à la même force. Sur `ssm` la moyenne est fixée par une seule graine ($+1.2141$ ; les
quatre autres : $+0.0016$, $-0.0005$, $+0.2412$, $-0.0384$, deux d'entre elles à moins de
$10^{-3}$ de zéro), pour une raison déclarée **avant** les runs (`§27.6`) : `ssm_k` échoue la
borne triviale arm-neutre sur 5 graines sur 5, `min_ssm` sur 3 sur 5 — la colonne `ssm` compare
deux bras dont ni l'un ni l'autre n'a quitté de façon fiable les baselines.

**L'énoncé retenu, celui de `§27.9` et pas la première phrase de `§27.8.3`** : le négatif est
porté par les cinq graines sur `mlp` et sur `transformer`. Sur `ssm`, il n'est **pas** compté
comme une troisième confirmation indépendante — la colonne est non informative pour la raison
qui vient d'être nommée, pas silencieusement ignorée. Le négatif généralise à `mlp` et à
`transformer` ; il ne dit rien de plus sur `ssm`, et ce chapitre n'écrit jamais qu'il
généralise à un compte de backbones.

In [ ]:
fig3 = REPO / "assets" / "fig3_factorial_effects.png"
if fig3.exists():
    display(Image(str(fig3)))
else:
    print("assets/fig3_factorial_effects.png absent de ce clone — voir la table ci-dessus, "
          "chiffres identiques (PROTOCOL.md §27.8.3, §27.9).")

**Lecture de la figure.** Trois panneaux, un par backbone, la ligne du zéro comme clé de
lecture : un point sous zéro dirait que le bras structuré fait mieux, au-dessus qu'il fait
pire. Les cinq points par panneau sont les cinq graines, pas une moyenne qui les cache — c'est
la figure elle-même qui montre que la colonne `ssm` a deux points quasi nuls et un point
éloigné, plutôt que de laisser la moyenne seule le suggérer. Ce que la figure ne montre pas :
l'éligibilité de `§27.3`, ni les huit relations `indeterminate` de `§4.2` — elle porte
uniquement le contraste apparié, la seule lecture de cette grille libre du confondant de
taux.

In [ ]:
figE = REPO / "assets" / "figE_ssm_delta.png"
if figE.exists():
    display(Image(str(figE)))
else:
    print("assets/figE_ssm_delta.png absent de ce clone — voir §4.3 ci-dessus, "
          "memes chiffres (PROTOCOL.md §23.6, §24.0, §27.6, §27.8.2).")

**Lecture de la figure.** Deux panneaux, tous deux lus sur `results_ssm_delta/` seul — la
campagne C qui a déclaré, avant `results_primary_factorial/`, que la famille `ssm` diverge à
grand `delta_max`. À **gauche**, `train_loss` contre le nombre d'updates, en échelle
logarithmique sur l'axe des x : les arms à `delta_max = 10.0` s'arrêtent entre l'époque 28 et
56, les arms à `4·dt` tiennent les 8000 updates jusqu'au bout. `min_ssm` à `delta_max = 10.0`
n'a **aucune courbe** : il devient non fini dès l'époque 0, avant qu'une seule ligne ne soit
écrite — l'absence est elle-même la donnée, annotée plutôt que masquée. À **droite**, la norme
du gradient du tout premier pas d'optimiseur, par graine, en échelle logarithmique : pour
`ssm_k`, les deux niveaux de `delta_max` donnent le **même** premier pas, graine par graine, à
trois décimales près — la contrainte de rayon n'agit pas à l'initialisation, elle agit de
façon cumulative au fil de l'entraînement.

**Ce qu'elle ne montre pas.** Aucun chiffre d'une autre racine : la norme de gradient
$2.4\times10^{18}$ mesurée sur `results_probe_d/` (`§18.0`) n'apparaît pas ici — la mélanger
à ce panneau violerait la règle d'auto-suffisance des racines (`§23.0`, « no result is read
across roots »). C'est un second ancrage, indépendant de la table de `§4.3` : il montre *où*
`ssm_k` et `min_ssm` divergent, pas seulement *que* le contraste apparié sur `ssm` est
dominé par une graine.

## 5. La leçon de la base

La section 3 a laissé une question ouverte : pourquoi `on-stable-nolift` — la transition
structurée seule, sans lift ni projection — est-elle **pire** que `off` et que `on-stable`
tous les deux, alors que la règle de lecture de `§25.3` l'attendait entre les deux ? La réponse
tient en une observation en forme close, et elle change ce que « le prior est exact » veut
dire.

### 5.1 Une affirmation de base, pas de coordonnées

**L'intuition, en dimension 2.** Un bloc rotation-scaling est une forme précise sur la
**matrice**, pas seulement sur son spectre. Prenons la rotation pure
$G = \begin{pmatrix} 0 & -1\\ 1 & 0\end{pmatrix}$ (valeurs propres $\pm i$) : dans la base
$(e_1,e_2)$, $G$ est déjà de la forme rotation-scaling ($\alpha=0$, $\omega=1$). Changeons de
base par un cisaillement $T = \begin{pmatrix}1&1\\0&1\end{pmatrix}$ — pas une rotation, un
simple changement de coordonnées : $TGT^{-1} = \begin{pmatrix}1&-2\\1&-1\end{pmatrix}$. Le
spectre est inchangé ($\pm i$ : une similitude le préserve toujours), mais la matrice ne l'est
pas — sa diagonale n'est plus constante ($1 \neq -1$) — et sa distance au bloc le plus proche
vaut $\sqrt{2.5}\approx1.58$, pas zéro. **Un spectre representable ne suffit pas ; la base dans
laquelle on regarde compte.** C'est exactement la question que `§3.4` pose en dimension 4, sur
le générateur du système conducteur.

Le générateur vrai du système conducteur, en coordonnées d'état, est
$A = \begin{pmatrix} 0 & I \\ -K & -cI\end{pmatrix}$. À $c=0.10$, l'axe spectral de `§3.4`
(notebook 00 §3) lit $\delta(c) = 0.000$ : les quatre valeurs propres de $A$ forment deux
paires conjuguées, exactement ce qu'un bloc rotation-scaling peut porter — mais, comme
l'exemple ci-dessus le montre, cela ne dit encore rien de la base.

**La formule fermée du meilleur bloc.** Pour un appariement de coordonnées $(i,j)$ fixé, et le
bloc candidat $M = \begin{pmatrix}a & b\\ c & d\end{pmatrix}$ extrait de $A$ à ces indices, le
bloc rotation-scaling le plus proche au sens de Frobenius est

$$\boxed{\;\alpha = \frac{a+d}{2}, \qquad \omega = \frac{c-b}{2}, \qquad
\text{résidu}_{\text{bloc}} = \frac{(a-d)^2+(b+c)^2}{2}\;}$$

où $\alpha$ moyenne les deux termes diagonaux (le taux de décroissance commun), $\omega$
moyenne l'antisymétrie hors diagonale (la fréquence de rotation commune), et le résidu de bloc
est ce que cette moyenne laisse derrière elle. Ce n'est pas un choix arbitraire : minimiser
$\|M-B\|_F$ sur les matrices $B=\begin{pmatrix}\alpha&-\omega\\\omega&\alpha\end{pmatrix}$ est
un problème de moindres carrés séparable en $\alpha$ et $\omega$, et cette moyenne en est la
solution (`PROTOCOL.md` §26.0). Toute entrée **hors** des blocs diagonaux doit s'annuler pour
qu'une matrice appartienne exactement à la classe ; si elle ne s'annule pas, elle compte pour sa
valeur entière dans la distance totale — c'est cette règle, appliquée à $(i,j)=(1,2)$ dans
l'exemple ci-dessus, qui donne $1.58$ et non $0$.

`on-stable-nolift` impose $\psi=\pi=\mathrm{identité}$ : la transition doit prendre cette forme
précise **dans la base où $A$ est écrite**, l'identité — pas dans une base au choix.

**[DÉMONTRÉ -> vérifié]** Reproduisons cette distance nous-mêmes, sur le générateur du système
conducteur, avant de la confronter au calcul du dépôt.

In [ ]:
# --- [DEMONTRE -> verifie] Distance de A a la classe rotation-scaling, deux bases -----------
from sld.systems.coupled_oscillators import CoupledOscillators

A = CoupledOscillators(c=0.10).A_mat        # notebook 02 §4.1 a deja verifie A_state == A_mat
PAIRINGS = [((0, 1), (2, 3)), ((0, 2), (1, 3)), ((0, 3), (1, 2))]


def nearest_block_rs(M, pairing):
    '''Le meilleur bloc-diagonal rotation-scaling pour un appariement FIXE (formule de §26.0) :
    tout ce qui est hors bloc est mis a zero et compte donc en entier dans l'ecart.'''
    B = np.zeros_like(M)
    for (i, j) in pairing:
        alpha = (M[i, i] + M[j, j]) / 2.0
        omega = (M[j, i] - M[i, j]) / 2.0
        B[i, i] = B[j, j] = alpha
        B[i, j], B[j, i] = -omega, omega
    return B


def identity_basis_distance(M):
    '''Distance minimale sur les 3 appariements possibles a 4 coordonnees.'''
    dists = {p: np.linalg.norm(M - nearest_block_rs(M, p)) for p in PAIRINGS}
    best_p = min(dists, key=dists.get)
    return dists[best_p], best_p


d_id, p_id = identity_basis_distance(A)
norm_A = np.linalg.norm(A)
print(f"distance en base IDENTITE : {d_id:.4f}  (appariement {p_id})")
print(f"||A||_F = {norm_A:.4f}  ->  {100 * d_id / norm_A:.1f}% de ||A||_F")

# base de Jordan reelle : deux paires conjuguees -> deux blocs 2x2 exacts (memes formules)
lam, vec = np.linalg.eig(A)
assert np.all(np.abs(lam.imag) > 1e-9), "les 4 valeurs propres doivent etre complexes a c=0.10"
cols, used = [], set()
for i in range(len(lam)):
    if i in used:
        continue
    j = int(np.argmin([abs(lam[k] - np.conj(lam[i])) if k != i else np.inf for k in range(len(lam))]))
    used |= {i, j}
    cols += [vec[:, i].real, vec[:, i].imag]
P_basis = np.column_stack(cols)
A_jordan = np.linalg.solve(P_basis, A @ P_basis)          # T A T^-1, T = P_basis^-1
d_best, p_best = identity_basis_distance(A_jordan)
print(f"\ndistance en base de JORDAN REELLE : {d_best:.3e}  (appariement {p_best})")
print("blocs :", np.round(np.diag(A_jordan)[[0, 2]], 4), "+-i",
      np.round([A_jordan[1, 0], A_jordan[3, 2]], 4))

**Lecture.** En base identité, $A$ est à $1.0654$ de la classe rotation-scaling — soit
**36%** de $\|A\|_F = 2.9614$. Dans sa base de Jordan réelle, la même classe est à
$1.97\times10^{-15}$ : essentiellement zéro. Ces deux nombres et les blocs
$(-0.05,\pm1.5187)$, $(-0.05,\pm1.0902)$ reproduisent exactement `PROTOCOL.md` §26.0.

`psi` et `pi` ne sont donc pas une surcharge posée sur la transition structurée : **ils sont le
changement de base que la structure exige.** `on-stable-nolift` retire ce changement de base et
force la transition à travailler dans la seule base où $\delta$ redevenait faussement gratuit —
l'identité — et c'est pour cela qu'il est pire que les deux arms, pas malgré une propriété de
l'optimiseur.

*(Précaution méthodologique, `§26.0` la formule explicitement : $\delta(c)$ — l'axe spectral de
`notebook 00` §3 — est une distance **spectrale**, celle ci-dessus est une distance de
**Frobenius sur le générateur**. Les deux ne sont pas commensurables terme à terme ; l'argument
tient à l'intérieur d'**une seule** métrique, comparée à elle-même entre les deux bases, au
même $c$.)*

In [ ]:
# --- [PARITE PACKAGE] La meme distance, via src/sld/systems/representability.py -------------
from sld.systems.representability import real_modal_distance, representability_distance

d_pkg, pairing_pkg = representability_distance(A)
d_pkg_best = real_modal_distance(A)

print(f"identite  : notebook {d_id:.10f}  vs  sld {d_pkg:.10f}  "
      f"-> ecart {abs(d_id - d_pkg):.2e}")
print(f"Jordan    : notebook {d_best:.3e}  vs  sld {d_pkg_best:.3e}  "
      f"-> ecart {abs(d_best - d_pkg_best):.2e}")

assert np.isclose(d_id, d_pkg, atol=1e-10)
assert np.isclose(d_best, d_pkg_best, atol=1e-12)
assert abs(round(d_id, 4) - 1.0654) < 1e-3
print("\nparite confirmee : representability_distance et real_modal_distance retombent sur "
      "les memes chiffres que le calcul minimal ci-dessus, et sur PROTOCOL.md §26.0.")

**Lecture.** L'implémentation minimale et `sld.systems.representability` — le module que
`assets/figG_decomposition.png` interroge pour annoter sa propre figure — tombent sur les mêmes
dix décimales en base identité et sur le même ordre de grandeur en base de Jordan. La parité
n'est pas une coïncidence : les deux calculs appliquent la même formule fermée.

### 5.2 La figure du dépôt, et ce qu'elle ajoute

In [ ]:
figG = REPO / "assets" / "figG_decomposition.png"
if figG.exists():
    display(Image(str(figG)))
else:
    print("assets/figG_decomposition.png absent de ce clone — la table de §3.3 et les chiffres "
          "de §5.1 portent le meme resultat (PROTOCOL.md §25.3, §26.0).")

**Lecture.** Le panneau principal montre `off`, `on-stable` et `on-stable-nolift` contre
`n_episodes` $\in\{8, 256\}$ sur `results_osc_decomp/` : `on-stable-nolift` reste au-dessus des
deux autres bras aux deux tailles, $3.1\times$ pire que `on-stable` à $n=8$ et $13.1\times$ à
$n=256$. L'encart reproduit les deux chiffres de `§5.1` — $1.0654$ (36% de $\|A\|_F$) en base
identité, $1.97\times10^{-15}$ en meilleure base — comme l'explication, pas comme une
coïncidence de plus. Ce que la figure ne montre pas : deux confondants déclarés avant les runs
et non retirés — `on-stable-nolift` porte $15.7\%$ de paramètres en moins par construction
(`§23.2` gèle le backbone plutôt que d'apparier le budget) et hérite le taux de `on-stable`
faute d'un second balayage autorisé (`§23.5(2)`). Ni l'un ni l'autre n'explique une **inversion
de l'ordre attendu** — seule la géométrie de la base le fait.

## 6. Ce qu'on ne sait pas

Ce registre n'est pas une clause de style : un dossier qui se vend sur l'honnêteté
méthodologique se juge ici. Il reprend le ledger ouvert au `notebook 00` et le met à jour avec
tout ce que ce chapitre vient de mesurer.

### 6.1 Le contraste ON/OFF a une mesure, et une racine en donne le mécanisme — sans y répondre entièrement

Le contraste appairé de `§4.3` est porté par `mlp` et par `transformer` ; `ssm` reste non
informatif pour la raison déclarée en `§27.9`. La leçon de la base (`§5`) désigne un candidat
mécanistique : la pénalité vient de la **découverte jointe** de la base et de l'opérateur, pas
de la classe structurée en elle-même. `§26.0` s'arrête volontairement avant de le tester —
*« No experiment is declared to test this »*.

Une racine plus tardive le teste, en donnant la vraie base à l'arm structuré au lieu de la
laisser la découvrir (`PROTOCOL.md` §28, `results_basis/`, 15 cellules). **[MESURÉ —
`PROTOCOL.md` §28.7]** donner la base retire la majorité de la pénalité sur chaque graine (P2,
0+/5−) et ramène l'arm au gate de contrôle positif que la version à base apprise échoue sur
les cinq graines (5/5 `learns` contre 0/5) — mais elle ne met pas la classe structurée devant
la classe non structurée (P3, à cheval sur zéro, 3+/2−). L'énoncé retenu, celui que `§28.4`
avait fixé avant de voir un seul chiffre : *« the coupling is a contributing mechanism; it is
not sufficient to reverse the sign »*. Le détail de cette expérience — le protocole complet,
les trois arms, les trois prédictions — est celui du `notebook 05`.

Ce qui reste hors de portée après cette racine : aucun test équivalent sur `transformer` ou
sur `ssm`, aucun sur `DampedPendulum` — `§21.4` tient sans changement, **M4 n'est pas
franchi**, et rien dans ce chapitre n'étend une conclusion de `CoupledOscillators` au régime
non linéaire.

### 6.2 Deux axes ont couru depuis la dernière fois que ce ledger a été écrit — un troisième ne courra plus sans un objecteur nommé

L'axe taille d'échantillon et l'axe de mauvaise spécification, purement prédictifs au
`notebook 00`, ont tourné (`§23-§25`) : la structure coûte à chaque taille d'échantillon
mesurée, et le ratio de coût croît avec la mauvaise spécification du prior — mais, `§26.1` le
précise, porté par l'amélioration du bras `off` plus que par une dégradation du bras structuré,
une nuance que ce chapitre ne développe pas plus loin. Ce qui reste fermé : un troisième
niveau de `h_fr`, un cinquième niveau de mauvaise spécification, un second balayage de taux —
`§23.5` et `§25.5` l'interdisent explicitement, quel que soit le résultat. Et la porte
elle-même s'est refermée un cran plus loin : après deux levées successives du même plafond
(`§25.5` puis `§27.5`), toute racine suivante exige désormais **une objection écrite d'un
lecteur nommé, qui n'est pas l'auteur** (`§28.5`) — une barre délibérément plus haute que celle
que cette section vient de franchir.

In [ ]:
figC = REPO / "assets" / "figC_sample_size.png"
if figC.exists():
    display(Image(str(figC)))
else:
    print("assets/figC_sample_size.png absent de ce clone — voir §6.2 ci-dessus, "
          "memes chiffres (PROTOCOL.md §23.5).")

**Lecture de la figure.** L'axe `n_episodes`, campagne A (`results_osc_n/`, racine de `§3.2`).
Chaque panneau lit `m_low` par cellule : les cinq graines en points, la médiane en barre, et un
indicateur de dispersion explicite — la dispersion est le résultat, pas un habillage :
`on-free` à $n=8$ s'étend de $1.18$ à $8.57$, une médiane seule aurait affiché « $1.78$ » et
caché un facteur sept. L'estimande pré-enregistré (`§23.5`) est le **signe** de l'effet, jamais
un seuil de passage — ces racines n'ont pas de gate (`§23.4`).

**Ce qu'elle ne montre pas.** Un pass/fail : aucune de ces cellules n'est jugée contre un seuil,
seulement lue pour son signe et son ordre de grandeur. La figure ne dit rien non plus sur *quel*
mécanisme produit l'écart — c'est la question que `§5` reprend.

In [ ]:
figD = REPO / "assets" / "figD_misspecification.png"
if figD.exists():
    display(Image(str(figD)))
else:
    print("assets/figD_misspecification.png absent de ce clone — voir §6.2 ci-dessus, "
          "memes chiffres (PROTOCOL.md §23.5, §23.7, §26.1).")

**Lecture de la figure.** L'axe $\delta(c)$, campagne B (`results_osc_delta/`, racine de
`§3.2`), en forme close (`§3.4` de `notebook 00`). Deux panneaux, et l'ordre compte : à
**gauche**, les absolus — `m_low` des trois arms et les deux baselines triviales qui
normalisent la métrique, sur les mêmes axes ; à **droite**, le ratio seul, jamais montré sans
le panneau de gauche à côté. Le ratio confirme le signe pré-enregistré ($+9.61$ pour `on-free`,
$+10.52$ pour `on-stable` par unité de $\delta$) — mais le panneau de gauche montre que les
arms structurés restent **épinglés** près de $m_{\text{low}}\sim1$ sur tout l'axe ($\times1.20$
pour `on-free`, $\times1.24$ pour `on-stable`), pendant que `off` s'améliore $\times3.67$ : la
croissance du ratio est un effet de **dénominateur**, porté par `off`, pas une dégradation du
bras structuré.

**Ce qu'elle ne montre pas.** Que le mécanisme prédit ait été le bon — `§26.1` le nomme
explicitement : « the predicted sign is confirmed and its mechanism is not the one the
prediction supposed ». Le signe pré-enregistré tient ; ce que le ratio seul aurait laissé
croire sur son origine ne tient pas.

### 6.3 L'effet de la dégénérescence de jauge sur la descente de gradient reste non mesuré

`notebook 02` §3 démontre $2r$ directions de perte exactement plates, vérifiées à la précision
machine, et sa contrainte $\pi=\psi^+$ répare l'aller-retour $z\to h\to z$ sans supprimer ni la
jauge d'échelle ni la jauge de rotation. Ce que cette jauge coûte à l'optimisation reste
**inconnu** : ni bras avec terme de reconstruction, ni bras avec $\pi=\psi^+$ imposé pendant
l'entraînement, ni spectre du hessien mesuré. La leçon de `§5` montre que $\psi/\pi$ portent un
travail géométrique nécessaire ; elle ne dit rien de la difficulté que ce travail ajoute à la
descente de gradient elle-même.

### 6.4 Le régime d'entraînement : partiellement mesuré, jamais essayé en curriculum

`§2.5` a montré, sur une maquette, qu'un rollout fermé de 25 pas depuis l'initialisation suffit
à créer un écart de treize ordres de grandeur. `§3.3` a mesuré, sur le vrai système, que le
signe du négatif **survit** au changement d'horizon ($h_fr \in \{5, 25\}$) — la maquette avait
raison sur l'ampleur de l'effet, elle n'expliquait pas à elle seule le signe, qui tient plus
largement. Ce qui n'a jamais tourné : un curriculum à horizon croissant, un *teacher forcing*
partiel, une borne sur $r$ pendant l'entraînement plutôt qu'après. `§25.1` les distingue
explicitement de la question des deux niveaux fixes et les laisse pour une autre fois.

### 6.5 Ce que ce chapitre lui-même n'établit pas

Les sections 2.3 à 2.6 sont des simulations **[ILLUSTRATIVES]** : des maquettes où tout, sauf
un ingrédient, est contrôlé. Elles ne reproduisent aucun run du dépôt et aucune de leurs
valeurs numériques ne doit être citée comme un résultat mesuré. La leçon de base (`§5`) tient à
l'intérieur d'une seule métrique comparée à elle-même — la confondre avec `§3.4`'s $\delta(c)$
spectral serait une erreur d'unité, pas une nuance.

## 7. Ce qui a été prouvé ici, et le pont

**Prouvé.** L'identifiabilité d'abord : sur le système conducteur, la solution exacte est dans
la classe et récupérable par moindres carrés à $10^{-8}$, et la descente de gradient déclarée y
atterrit environ $200\times$ plus loin — quatre explications concurrentes testées et éliminées,
et une maquette qui isole le rollout fermé comme ingrédient suffisant à un écart de treize
ordres de grandeur. Le chemin des campagnes : le pendule ne franchit pas son gate (une cellule
sur huit, `§21.4`), le dépôt pivote par dérogation écrite vers le seul système où le prior est
vrai, et un cycle de 164 cellules y mesure un négatif qui survit au changement d'horizon
d'entraînement. Le factoriel primaire referme le trou d'un seul backbone : sous la règle
pré-enregistrée, une relation contredite et huit indéterminées — la règle n'a pas été réparée
après coup — et, sur le seul contraste libre de confondant, un négatif porté par `mlp` et par
`transformer`. La leçon de la base : le prior à $\delta=0$ est une affirmation de base, pas de
coordonnées — $1.0654$ (36% de $\|A\|_F$) en identité, $1.97\times10^{-15}$ en base de Jordan
réelle, et $\psi/\pi$ sont ce changement de base, pas une surcharge dessus.

**Lien avec `sld`.** `src/sld/systems/coupled_oscillators.py` (`CoupledOscillators.A_mat`,
parité déjà établie au `notebook 02` §4.1), `src/sld/systems/representability.py`
(`representability_distance`, `real_modal_distance`, `real_modal_basis`), `PROTOCOL.md` §20
à §28 pour chaque chiffre cité ici, `assets/fig3_factorial_effects.png` et
`src/sld/viz/fig3_factorial_effects.py`, `assets/figG_decomposition.png` et
`src/sld/viz/figC_sample_size.py::make_figG`.

**La limite honnête.** Le registre de la section 6 : un mécanisme mesuré mais pas suffisant
(P3 indéterminée), deux axes fermés à toute extension sans objecteur nommé, une jauge dont
l'effet sur l'optimisation reste non mesuré, un régime d'entraînement mesuré sur deux niveaux
seulement, jamais en curriculum.

**Le pont.** La leçon de la base pose une question précise, que ce chapitre nomme sans
l'épuiser : si la pénalité vient de la découverte **jointe** de la base et de l'opérateur, que
se passe-t-il si on **donne** la base ? Le `notebook 05` déroule cette expérience oracle
(`PROTOCOL.md` §28) de bout en bout — le protocole, les trois arms, les trois prédictions
écrites avant les quinze cellules, et la figure `figH`.